In [1]:
#%%capture
%pip install --index-url https://pypi.org/simple pandas numpy \
    scipy matplotlib seaborn ipython \
    torch transformers scikit-learn tqdm \
    peft ipywidgets

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 5.4 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 10.9 MB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 12.5 MB/s eta 0:00:01
     |████████████████████████████████| 7.8 MB 8.3 MB/s eta 0:00:01
     |████████████████████████████████| 294 kB 9.5 MB/s eta 0:00:01
     |████████████████████████████████| 73.6 MB 120 kB/s  eta 0:00:01     |████████████▉                   | 29.4 MB 8.4 MB/s eta 0:00:06     |█████████████▎                  | 30.5 MB 8.4 MB/s eta 0:00:06
     |████████████████████████████████| 12.0 MB 10.0 MB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 919 kB/s eta 0:00:01    |████████████▌                   | 4.4 MB 9.7 MB/s eta 0:00:01
     |████████████████████████████████| 80 kB 10.0 MB/s eta 0:00:01
     |████████████████████████████████| 504 kB 10.5 MB/s eta 0:00:01
     |████████████████████████

In [2]:
from collections.abc import Sequence
from dataclasses import dataclass
import os

import numpy as np
import pandas as pd
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, EsmForSequenceClassification

# --- GLOBAL CONFIGURATION ---

# Test mode flag: If True, uses a subset of data and fewer epochs.
TEST_MODE: bool = True


@dataclass(frozen=True)
class Config:  # pylint: disable=too-many-instance-attributes
    """Global configuration settings."""

    dataset_path: str = "data/processed/training_data.csv"
    random_state: int = 42
    test_fraction: float = 0.2
    default_batch_size: int = 2 if TEST_MODE else 32
    default_epochs: int = 1 if TEST_MODE else 3
    base_model_name: str = "facebook/esm2_t6_8M_UR50D"
    large_model_name: str = "facebook/esm2_t12_35M_UR50D"
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    models_dir: str = "models"


GLOBAL_CONFIG: Config = Config()
print(f"TEST_MODE is set to: {TEST_MODE}")
print(f"Running on {GLOBAL_CONFIG.device}")

# Set seeds for reproducibility
torch.manual_seed(GLOBAL_CONFIG.random_state)
np.random.seed(GLOBAL_CONFIG.random_state)

TEST_MODE is set to: True
Running on cpu


## 1. Domain Functions: Data Preparation & Dataloaders

In [6]:
class EpitopeDataset(Dataset):
    """Custom Dataset for Epitope Sequences and Labels."""

    def __init__(self, sequences: Sequence[str], labels: Sequence[float]):
        self.sequences = sequences
        self.labels = labels

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, idx: int) -> tuple[str, int]:
        sequence = str(self.sequences[idx])
        label = int(self.labels[idx])
        return sequence, label


def create_collate_fn(tokenizer: AutoTokenizer):
    """Creates a collate function that wraps the tokenizer."""

    def collate_fn(batch: Sequence[tuple[str, int]]) -> tuple[torch.Tensor, torch.Tensor]:
        sequences, labels = zip(*batch)
        tokenized_sequences_tensor = tokenizer(
            list(sequences), return_tensors="pt", padding=True, truncation=True
        )
        labels_tensor = torch.tensor(list(labels), dtype=torch.long)
        return tokenized_sequences_tensor, labels_tensor

    return collate_fn

In [7]:
def load_and_split_data(
    dataset_path: str, *, test_fraction: float, random_state: int, batch_size: int
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Loads data, handles splitting, and optionally subsets for testing."""
    data_df = pd.read_csv(dataset_path)

    train_df, test_df = train_test_split(
        data_df,
        test_size=test_fraction,
        random_state=random_state,
        stratify=data_df["is_binded"],
    )

    if TEST_MODE:
        # Subset to just a few batches worth of data
        subset_size = 4 * batch_size
        train_df = train_df.iloc[:subset_size]
        test_df = test_df.iloc[:subset_size]

    return train_df, test_df


def prepare_dataloaders(
    *, train_df: pd.DataFrame, test_df: pd.DataFrame, model_name: str, batch_size: int
) -> tuple[DataLoader, DataLoader]:
    """Instantiates the Datasets and DataLoaders."""
    train_sequences = train_df["sequence"].tolist()
    train_labels = train_df["is_binded"].tolist()

    test_sequences = test_df["sequence"].tolist()
    test_labels = test_df["is_binded"].tolist()

    train_dataset = EpitopeDataset(train_sequences, train_labels)
    test_dataset = EpitopeDataset(test_sequences, test_labels)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    collate_fn = create_collate_fn(tokenizer)

    train_dataloader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn
    )
    test_dataloader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn
    )

    return train_dataloader, test_dataloader

In [8]:
TRAIN_DF, TEST_DF = load_and_split_data(
    GLOBAL_CONFIG.dataset_path,
    test_fraction=GLOBAL_CONFIG.test_fraction,
    random_state=GLOBAL_CONFIG.random_state,
    batch_size=GLOBAL_CONFIG.default_batch_size,
)
print(f"Train DF shape: {TRAIN_DF.shape}")
print(f"Test DF shape: {TEST_DF.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/training_data.csv'

In [5]:
TRAIN_BASE_MODEL_DATALOADER, TEST_BASE_MODEL_DATALOADER = prepare_dataloaders(
    train_df=TRAIN_DF,
    test_df=TEST_DF,
    model_name=GLOBAL_CONFIG.base_model_name,
    batch_size=GLOBAL_CONFIG.default_batch_size,
)

NameError: name 'prepare_dataloaders' is not defined

## 3. Training and Evaluation Functions

In [11]:
def train_model(
    *,
    model: torch.nn.Module,
    train_dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    num_epochs: int,
    desc_prefix: str = "Epoch",
) -> None:  # pylint: disable=too-many-arguments
    """Executes the training loop for the provided model and optimizer."""
    model.train()

    print(f"Starting training on {device}...")

    for epoch in range(num_epochs):
        total_loss = 0.0
        progress_bar = tqdm(train_dataloader, desc=f"{desc_prefix} {epoch + 1}")

        for inputs in progress_bar:
            sequences, labels = inputs
            sequences = sequences.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(**sequences, labels=labels)
            loss = outputs.loss

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_dataloader)
        print(f"{desc_prefix} {epoch + 1} finished. Average Loss on {epoch} epoch: {avg_loss:.4f}")

    print("Training complete.\n")


def evaluate_model(
    model: torch.nn.Module,
    test_dataloader: DataLoader,
    device: torch.device,
    model_name_label: str = "Model",
) -> dict[str, float]:
    """Evaluates the model on the given dataloader and calculates metrics."""
    model.eval()

    all_predictions = []
    all_true_labels = []
    all_predicted_probs = []

    with torch.no_grad():
        for inputs in test_dataloader:
            sequences, labels = inputs
            sequences = sequences.to(device)

            outputs = model(**sequences)
            logits = outputs.logits

            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            predictions = torch.argmax(logits, dim=1).cpu().numpy()

            all_predicted_probs.extend(probs)
            all_predictions.extend(predictions)
            all_true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_true_labels, all_predictions)
    precision = precision_score(all_true_labels, all_predictions, average="binary", zero_division=0)
    recall = recall_score(all_true_labels, all_predictions, average="binary", zero_division=0)

    try:
        auc_score = roc_auc_score(all_true_labels, all_predicted_probs)
    except ValueError:
        auc_score = 0.0

    print(f"--- Performance: {model_name_label} ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"AUC Score: {auc_score:.4f}\n")

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "auc_score": float(auc_score),
    }

In [12]:
def load_model(
    model_name: str, device: torch.device, weights_path: str | None = None
) -> EsmForSequenceClassification:
    """Loads a pre-trained ESM model for sequence classification, optionally loading weights."""
    model = EsmForSequenceClassification.from_pretrained(model_name, num_labels=2)
    if weights_path is not None:
        model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    return model


def save_model(model: torch.nn.Module, file_path: str) -> None:
    """Saves the model weights (state dict) to the specified file path."""
    output_dir = os.path.dirname(file_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    torch.save(model.state_dict(), file_path)
    print(f"Model weights saved to {file_path}")

## 3. Training of different setups of base model

In [15]:
def show_non_trained_performance() -> None:
    """Evaluates baseline performance of the un-fine-tuned base model."""
    non_trained_model = load_model(GLOBAL_CONFIG.base_model_name, GLOBAL_CONFIG.device)
    _ = evaluate_model(
        non_trained_model,
        TEST_BASE_MODEL_DATALOADER,
        GLOBAL_CONFIG.device,
        "Initial Base Model (No training)",
    )


show_non_trained_performance()

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Performance: Initial Base Model (No training) ---
Accuracy:  0.6250
Precision: 0.0000
Recall:    0.0000
AUC Score: 0.1333



In [16]:
def run_head_only_experiment() -> None:
    """Fine-tunes only the classifier head while keeping the ESM backbone frozen."""
    print("=== Experiment 1: Head-Only Fine-Tuning ===")
    model_head_only = load_model(GLOBAL_CONFIG.base_model_name, GLOBAL_CONFIG.device)

    # Train only classifier head
    for param in model_head_only.esm.parameters():
        param.requires_grad = False
    for param in model_head_only.classifier.parameters():
        param.requires_grad = True
    optimizer_head_only = optim.AdamW(model_head_only.parameters(), lr=5e-5)

    train_model(
        model=model_head_only,
        train_dataloader=TRAIN_BASE_MODEL_DATALOADER,
        optimizer=optimizer_head_only,
        device=GLOBAL_CONFIG.device,
        num_epochs=GLOBAL_CONFIG.default_epochs,
        desc_prefix="Head-Only Epoch",
    )

    _ = evaluate_model(
        model_head_only,
        TEST_BASE_MODEL_DATALOADER,
        GLOBAL_CONFIG.device,
        "Head-Only Fine-Tuned",
    )
    save_model(model_head_only, f"{GLOBAL_CONFIG.models_dir}/head_only.pt")


run_head_only_experiment()

=== Experiment 1: Head-Only Fine-Tuning ===


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training on cpu...


Head-Only Epoch 1:   0%|          | 0/4 [00:00<?, ?it/s]

Head-Only Epoch 1 finished. Average Loss on 0 epoch: 0.6691
Training complete.

--- Performance: Head-Only Fine-Tuned ---
Accuracy:  0.6250
Precision: 0.0000
Recall:    0.0000
AUC Score: 0.8000



In [17]:
def run_full_fit_experiment() -> None:
    """Fine-tunes all model parameters using a uniform learning rate."""
    print("=== Experiment 2: Full Fine-Tuning (Uniform LR) ===")
    model_full_ft = load_model(GLOBAL_CONFIG.base_model_name, GLOBAL_CONFIG.device)

    # Ensure we train all params
    for param in model_full_ft.parameters():
        param.requires_grad = True
    optimizer_full_ft = optim.AdamW(model_full_ft.parameters(), lr=1e-5)

    train_model(
        model=model_full_ft,
        train_dataloader=TRAIN_BASE_MODEL_DATALOADER,
        optimizer=optimizer_full_ft,
        device=GLOBAL_CONFIG.device,
        num_epochs=GLOBAL_CONFIG.default_epochs,
        desc_prefix="Full FT Epoch",
    )

    _ = evaluate_model(
        model_full_ft,
        TEST_BASE_MODEL_DATALOADER,
        GLOBAL_CONFIG.device,
        "Fully Fine-Tuned (Uniform LR)",
    )
    save_model(model_full_ft, f"{GLOBAL_CONFIG.models_dir}/full_ft.pt")


run_full_fit_experiment()

=== Experiment 2: Full Fine-Tuning (Uniform LR) ===


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training on cpu...


Full FT Epoch 1:   0%|          | 0/4 [00:00<?, ?it/s]

Full FT Epoch 1 finished. Average Loss on 0 epoch: 0.6926
Training complete.

--- Performance: Fully Fine-Tuned (Uniform LR) ---
Accuracy:  0.6250
Precision: 0.0000
Recall:    0.0000
AUC Score: 0.4667



In [ ]:
def run_differential_lr_experiment() -> None:
    """Fine-tunes the full model using differential learning rates for backbone and head."""
    print("=== Experiment 3: Full Fine-Tuning (Differential LR) ===")
    model_diff_lr = load_model(GLOBAL_CONFIG.base_model_name, GLOBAL_CONFIG.device)

    # Set bigger lr to classifier head
    for param in model_diff_lr.parameters():
        param.requires_grad = True
    grouped_parameters = [
        {"params": model_diff_lr.esm.parameters(), "lr": 2e-5},
        {"params": model_diff_lr.classifier.parameters(), "lr": 1e-3},
    ]
    optimizer_diff_lr = optim.AdamW(grouped_parameters)

    train_model(
        model=model_diff_lr,
        train_dataloader=TRAIN_BASE_MODEL_DATALOADER,
        optimizer=optimizer_diff_lr,
        device=GLOBAL_CONFIG.device,
        num_epochs=GLOBAL_CONFIG.default_epochs,
        desc_prefix="Diff LR Epoch",
    )

    _ = evaluate_model(
        model_diff_lr,
        TEST_BASE_MODEL_DATALOADER,
        GLOBAL_CONFIG.device,
        "Fully Fine-Tuned (Differential LR)",
    )
    save_model(model_diff_lr, f"{GLOBAL_CONFIG.models_dir}/diff_lr.pt")


run_differential_lr_experiment()

=== Experiment 3: Full Fine-Tuning (Differential LR) ===


Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting training on cpu...


Diff LR Epoch 1: 100%|██████████| 4/4 [00:00<00:00, 19.12it/s, loss=0.4158]

Diff LR Epoch 1 finished. Average Loss on 0 epoch: 0.6828
Training complete.

--- Performance: Fully Fine-Tuned (Differential LR) ---
Accuracy:  0.6250
Precision: 0.0000
Recall:    0.0000
AUC Score: 0.7333



In [155]:
def run_lora_ft_experiment() -> None:
    """Fine-tunes the base model using Low-Rank Adaptation (LoRA)."""
    print("=== Experiment 4: LoRA Fine-Tuning ===")
    base_lora_model = load_model(GLOBAL_CONFIG.base_model_name, GLOBAL_CONFIG.device)

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["query", "key", "value"],
    )

    model_lora = get_peft_model(base_lora_model, lora_config)
    model_lora.print_trainable_parameters()
    optimizer_lora = optim.AdamW(model_lora.parameters(), lr=1e-4)

    train_model(
        model=model_lora,
        train_dataloader=TRAIN_BASE_MODEL_DATALOADER,
        optimizer=optimizer_lora,
        device=GLOBAL_CONFIG.device,
        num_epochs=GLOBAL_CONFIG.default_epochs,
        desc_prefix="LoRA Epoch",
    )

    _ = evaluate_model(
        model_lora, TEST_BASE_MODEL_DATALOADER, GLOBAL_CONFIG.device, "LoRA Fine-Tuned"
    )
    save_model(model_lora, f"{GLOBAL_CONFIG.models_dir}/lora.pt")


run_lora_ft_experiment()

=== Experiment 4: LoRA Fine-Tuning ===


Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 195,522 || all params: 8,036,285 || trainable%: 2.4330
Starting training on cpu...


Diff LR Epoch 1: 100%|██████████| 4/4 [00:00<00:00, 26.19it/s, loss=0.7970]


Diff LR Epoch 1 finished. Average Loss on 0 epoch: 0.7003
Training complete.

--- Performance: LoRA Fine-Tuned ---
Accuracy:  0.6250
Precision: 0.0000
Recall:    0.0000
AUC Score: 0.6000



In [159]:
def run_large_model_experiment() -> None:
    """Fine-tunes a larger ESM-2 model (35M parameters) with differential learning rates."""
    print("=== Experiment 5: Larger Model Fine-Tuning ===")
    print(f"Initializing {GLOBAL_CONFIG.large_model_name}...")

    train_dataloader, test_dataloader = prepare_dataloaders(
        train_df=TRAIN_DF,
        test_df=TEST_DF,
        model_name=GLOBAL_CONFIG.large_model_name,
        batch_size=GLOBAL_CONFIG.default_batch_size,
    )

    model_large = load_model(GLOBAL_CONFIG.large_model_name, GLOBAL_CONFIG.device)

    _ = evaluate_model(
        model_large,
        test_dataloader,
        GLOBAL_CONFIG.device,
        "Large Model (Differential LR) (No Training)",
    )

    for param in model_large.parameters():
        param.requires_grad = True
    large_grouped_parameters = [
        {"params": model_large.esm.parameters(), "lr": 2e-5},
        {"params": model_large.classifier.parameters(), "lr": 1e-3},
    ]

    optimizer_large = optim.AdamW(large_grouped_parameters)

    train_model(
        model=model_large,
        train_dataloader=train_dataloader,
        optimizer=optimizer_large,
        device=GLOBAL_CONFIG.device,
        num_epochs=GLOBAL_CONFIG.default_epochs,
        desc_prefix="Large Model Epoch",
    )

    _ = evaluate_model(
        model_large, test_dataloader, GLOBAL_CONFIG.device, "Large Model (Differential LR)"
    )
    save_model(model_large, f"{GLOBAL_CONFIG.models_dir}/large_model.pt")


run_large_model_experiment()

=== Experiment 5: Larger Model Fine-Tuning ===
Initializing facebook/esm2_t12_35M_UR50D...


Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


--- Performance: Large Model (Differential LR) (No Training) ---
Accuracy:  0.3750
Precision: 0.3750
Recall:    1.0000
AUC Score: 0.7333

Starting training on cpu...


Large Model Epoch 1: 100%|██████████| 4/4 [00:00<00:00,  6.24it/s, loss=0.1155]


Large Model Epoch 1 finished. Average Loss on 0 epoch: 0.9033
Training complete.

--- Performance: Large Model (Differential LR) ---
Accuracy:  0.6250
Precision: 0.0000
Recall:    0.0000
AUC Score: 0.6667

